In [1]:
from pathlib import Path

REPO = Path(
    "/home/sagemaker-user/"
    "Beverage-Price-Prediction-AWS"
)

INFERENCE_DIR = REPO / "src" / "inference"

INFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

(INFERENCE_DIR / "__init__.py").touch()

print(INFERENCE_DIR)

/home/sagemaker-user/Beverage-Price-Prediction-AWS/src/inference


In [2]:
%%writefile /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/inference/requirements.txt
xgboost-cpu==2.1.4
joblib>=1.3,<2

Overwriting /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/inference/requirements.txt


In [3]:
%%writefile /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/inference/setup.py
from setuptools import setup, find_packages

setup(
    name="beverage-inference",
    version="1.0.0",
    py_modules=["inference"],
    packages=find_packages(),
)

Writing /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/inference/setup.py


In [4]:
from pathlib import Path

INFERENCE_DIR = Path(
    "/home/sagemaker-user/"
    "Beverage-Price-Prediction-AWS/"
    "src/inference"
)

print((INFERENCE_DIR / "requirements.txt").read_text())
print((INFERENCE_DIR / "setup.py").read_text())

xgboost-cpu==2.1.4
joblib>=1.3,<2

from setuptools import setup, find_packages

setup(
    name="beverage-inference",
    version="1.0.0",
    py_modules=["inference"],
    packages=find_packages(),
)



In [1]:
%%writefile /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/inference/inference.py

import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# --------------------------------------------------
# Allow access to sibling preprocessing package
# --------------------------------------------------
CODE_DIR = Path(__file__).resolve().parent

if str(CODE_DIR) not in sys.path:a
    sys.path.insert(0, str(CODE_DIR))

from preprocessing.feature_utils import transform_features


# --------------------------------------------------
# Load model
# --------------------------------------------------

def model_fn(model_dir):

    bundle_path = os.path.join(
        model_dir,
        "model_bundle.joblib"
    )

    bundle = joblib.load(bundle_path)

    return bundle


# --------------------------------------------------
# Deserialize request
# --------------------------------------------------

def input_fn(request_body, request_content_type):

    if request_content_type != "application/json":
        raise ValueError(
            f"Unsupported content type: "
            f"{request_content_type}"
        )

    if isinstance(
        request_body,
        (bytes, bytearray)
    ):
        request_body = request_body.decode("utf-8")

    payload = json.loads(request_body)

    # Supported formats:
    #
    # {"instances": [{...}, {...}]}
    #
    # OR
    #
    # {...single record...}
    #
    # OR
    #
    # [{...}, {...}]

    if isinstance(payload, dict):

        if "instances" in payload:
            records = payload["instances"]
        else:
            records = [payload]

    elif isinstance(payload, list):
        records = payload

    else:
        raise ValueError(
            "JSON payload must be a record, "
            "list of records, or contain "
            "'instances'."
        )

    if not records:
        raise ValueError(
            "No prediction records supplied."
        )

    return pd.DataFrame(records)


# --------------------------------------------------
# Predict
# --------------------------------------------------

def predict_fn(input_data, bundle):

    preprocessing = bundle["preprocessing"]
    model = bundle["model"]

    inverse_price_map = (
        bundle["inverse_price_map"]
    )

    # Apply EXACT preprocessing learned
    # during training.
    X = transform_features(
        input_data,
        preprocessing
    )

    predictions = model.predict(X)

    probabilities = model.predict_proba(X)

    model_classes = model.classes_

    results = []

    for prediction, probability_row in zip(
        predictions,
        probabilities
    ):

        predicted_class = int(prediction)

        probability_map = {
            inverse_price_map[int(class_id)]:
                float(probability)

            for class_id, probability
            in zip(
                model_classes,
                probability_row
            )
        }

        results.append({
            "predicted_class":
                predicted_class,

            "predicted_price_range":
                inverse_price_map[
                    predicted_class
                ],

            "confidence":
                float(
                    np.max(
                        probability_row
                    )
                ),

            "probabilities":
                probability_map,
        })

    return results


# --------------------------------------------------
# Serialize response
# --------------------------------------------------

def output_fn(
    prediction,
    accept
):

    if accept not in (
        "application/json",
        "*/*"
    ):
        raise ValueError(
            f"Unsupported accept type: {accept}"
        )

    response = {
        "predictions": prediction
    }

    return (
        json.dumps(response),
        "application/json"
    )

Overwriting /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/inference/inference.py


In [4]:
import boto3
import io
import json
import shutil
import sys
import tarfile
import pandas as pd


REGION = "ap-south-1"

MODEL_URI = (
    "s3://krushang-beverage-ml-2026/models/"
    "beverage-xgboost-production-20260904074049/"
    "output/model.tar.gz"
)

PROCESSED_URI = (
    "s3://krushang-beverage-ml-2026/"
    "processed/cleaned_survey_results.csv"
)

s3 = boto3.client(
    "s3",
    region_name=REGION
)

In [5]:
test_model_dir = Path(
    "/tmp/beverage_inference_test"
)

if test_model_dir.exists():
    shutil.rmtree(test_model_dir)

test_model_dir.mkdir(
    parents=True
)


bucket = "krushang-beverage-ml-2026"

model_key = (
    "models/"
    "beverage-xgboost-production-20260904074049/"
    "output/model.tar.gz"
)

model_bytes = s3.get_object(
    Bucket=bucket,
    Key=model_key
)["Body"].read()


with tarfile.open(
    fileobj=io.BytesIO(model_bytes),
    mode="r:gz"
) as tar:

    tar.extractall(
        test_model_dir,
        filter="data"
    )


print(
    "Model files:",
    list(test_model_dir.iterdir())
)

Model files: [PosixPath('/tmp/beverage_inference_test/model_bundle.joblib'), PosixPath('/tmp/beverage_inference_test/training_metadata.json'), PosixPath('/tmp/beverage_inference_test/xgboost_model.json')]


In [6]:
processed_key = (
    "processed/"
    "cleaned_survey_results.csv"
)

data_bytes = s3.get_object(
    Bucket=bucket,
    Key=processed_key
)["Body"].read()

df = pd.read_csv(
    io.BytesIO(data_bytes)
)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (29956, 18)
['respondent_id', 'age', 'gender', 'zone', 'occupation', 'income_levels', 'consume_frequency(weekly)', 'current_brand', 'preferable_consumption_size', 'awareness_of_other_brands', 'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel', 'packaging_preference', 'health_concerns', 'typical_consumption_situations', 'price_range', 'age_group']


In [7]:
SRC_PATH = str(REPO / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)


from inference import inference


bundle = inference.model_fn(
    str(test_model_dir)
)


sample = (
    df
    .drop(
        columns=["price_range"],
        errors="ignore"
    )
    .head(2)
    .copy()
)


# Make JSON-safe records
records = json.loads(
    sample.to_json(
        orient="records"
    )
)

request_body = json.dumps({
    "instances": records
})


input_data = inference.input_fn(
    request_body,
    "application/json"
)

predictions = inference.predict_fn(
    input_data,
    bundle
)

response, content_type = (
    inference.output_fn(
        predictions,
        "application/json"
    )
)


print("Content-Type:", content_type)
print(
    json.dumps(
        json.loads(response),
        indent=2
    )
)

Content-Type: application/json
{
  "predictions": [
    {
      "predicted_class": 1,
      "predicted_price_range": "100-150",
      "confidence": 0.9726226925849915,
      "probabilities": {
        "50-100": 1.666276148171164e-05,
        "100-150": 0.9726226925849915,
        "150-200": 0.02736065536737442,
        "200-250": 3.2431091145923574e-09
      }
    },
    {
      "predicted_class": 3,
      "predicted_price_range": "200-250",
      "confidence": 1.0,
      "probabilities": {
        "50-100": 7.20301531094425e-13,
        "100-150": 1.9776374819441972e-13,
        "150-200": 8.168514753492673e-10,
        "200-250": 1.0
      }
    }
  ]
}


/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
